# Introduction to Pandas, Part 2
---
<img src="Imperial_logo.png" align = "left" width=200>
 <br><br>
 
- Copyright (c) Antoine Jacquier, 2024. All rights reserved

- Author: Jack Jacquier <a.jacquier@imperial.ac.uk>

- Platform: Tested on Windows 10 with Python 3.9

 **Origins**
 
 `pandas` was originally developed by [Wes McKinney](https://wesmckinney.com) while at AQR Capital in 2007-2010.

# Introduction to DataFrames

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
%matplotlib inline

## Create a Dataframe with dates

We start by creating series of Gaussian distributed values.

In [ ]:
N = 10000 ## size of the dataframe

In [ ]:
xx = np.random.standard_normal((N, 5))
df = pd.DataFrame(xx)

In [ ]:
type(df)

In [ ]:
df.head() ## Print the top five rows

In [ ]:
df.tail() ## Print the bottom five rows

In [ ]:
df.columns = ["Series 1", "Series 2", "Series 3", "Series 4", "Series 5"]
## Or more concisely:
#df.columns = ["Series %s" % i  for i in range(1,6)]
df.head()

Note that the index (first column on the left) has no real meaning here. 
However, in practice, thinking or each other column as time series, it may represent dates. 
We re-index the whole dataframe with this in mind.

#### Specifying a start date

In [ ]:
startYear = '2018'
startMonth = '01'
startDay = '01'
startDate = startYear + "-" + startMonth + "-" + startDay ## string concatenation
print("Start date: ", startDate)

In [ ]:
dates = pd.date_range(startDate, periods=N, freq="D")
df.index = dates
df.index.name = 'Date'
print(df.head(3))
print(df.tail(3))

## Basic dataframe manipulations

In [ ]:
df.keys()

An Index is a `pandas` class for an immutable sequence, which is the basis for axes.

In [ ]:
print('Names of columns:', [d for d in df.keys().values])
## similarly df.columns

Overview of the data

In [ ]:
df.info()

Basic statistics

In [ ]:
df.describe()

## Accessing elements from a dataframe

A  DataFrame may be indexed like an array, specifying the row and column number using .iloc (index location).
The location can be called either by the index number or by its name.

In [ ]:
df.head()

In [ ]:
print("First row, second column: ", df.iloc[0, 1], " or ", df.loc[startDate, 'Series 2'])

In [ ]:
myRow = df.iloc[2,:]
print(myRow)
print("Type: ", type(myRow))

Equivalently:

In [ ]:
myRow = df.loc['2018-01-03']
print(myRow, type(myRow))

In [ ]:
print("Accessing elements by slicing over columns: ")
print(df.iloc[0, 1:3])
print(type(df.iloc[0, 1:3]))

In [ ]:
print("Accessing elements by slicing over rows: ")
print(df.iloc[10:14, 1])
print(type(df.iloc[10:14, 1]))

In [ ]:
print("Accessing elements by slicing over boths: ")
print(df.iloc[10:14, 1:3])
print(type(df.iloc[10:14, 1:3]))

Question: What are the types of the following objects:
- df.iloc[2]
- df.iloc[2].values
- [d for d in df.iloc[2]]

### `.at` and `.iat`

These two commands are very similar to `.loc` and `.iloc`, and are actually faster, but slicing is not permitted.

In [ ]:
%timeit df.iloc[10, 2]
%timeit df.iat[10, 2]

In [ ]:
# no slices allowed!
df.iat[10, 2:4]

### Extract elements with conditions

In [ ]:
print(df.shape)
mySeriesName = "Series 2"
df[df[mySeriesName] > 3.4]

Note that the series `df` has not been modified. Only a copy has been printed!!

In [ ]:
print(df.shape)
print(mySeriesName)
mySeries =  df[mySeriesName][df[mySeriesName] > 3.4]
print(mySeries.shape, mySeries.head(3))

We can return the series only as a list:

In [ ]:
print(mySeries.values)

### *** Now do Exercise 1 ***

### User-defined functions

In [ ]:
df = df
df.head()

Here, the original dataframe has been modified!

In [ ]:
def myFunction(x, param):
    return x - param

print(mySeriesName)

df[mySeriesName] = df[mySeriesName].apply(myFunction, args=(10,))
df.head()

## Numpy on dataframe

This is where the power of the DataFrame comes into the picture, namely that operations need to be performed element by element, 
but apply to a whole series (column) at once.

In [ ]:
myIndex = input("Series index")

In [ ]:
ser = 'Series ' + myIndex
print('Maximum on the first Series:', df[ser].max())
print('Date (index) where this maximum is attained:', df[ser].argmax()) ## deprecated
print('Date where this maximum is attained:', df[ser].idxmax(), df.index[df[ser].argmax()])
print('Mean of the first series:', df[ser].mean())

We can also obtain general results about the whole dataframe / each series in the dataframe:

In [ ]:
df.mean()

## Plotting Dataframes

##### Plot all the time series

In [ ]:
df.plot();

**WHAT IS WRONG WITH THE PLOT?**

##### Plot one column at a time

In [ ]:
df[ser].plot(legend=ser);

##### Changing the type of plots

In [ ]:
datePlot = '2018-01-03'
df.loc[datePlot].plot(kind='bar', title=datePlot); ## for one particular day

## Sorting DataFrames

In [ ]:
print('Data sorted by ' + ser)
df.sort_values(by=ser).head()

*WARNING: the dataframe df itself has remained unchanged though...*

In [ ]:
df.head()

## Renaming columns

In [ ]:
df.rename(columns={'Series 1': 'SPX', 
                   'Series 2': 'DAX', 
                   'Series 3': 'GOOGL', 
                   'Series 4': 'AAPL', 
                   'Series 5': 'VIX'})
df.head()

In [ ]:
df.columns = ['SPX', 'DAX', 'GOOGL', 'AAPL', 'VIX']
print(df.head(2))

### A note on copies on slices....

In [ ]:
df.head()

In [ ]:
val = df.iloc[1,0]
val

In [ ]:
df[df["SPX"] == val]["DAX"] = 0.
print(df.head(3))

In [ ]:
df.loc[df["SPX"] == val, "DAX"] = 0.
print(df.head(3))

### *** Now do Exercise 2 ***

# Working with real data

## Financial data from Yahoo Finance

In [ ]:
# ! pip install pandas_datareader

In [ ]:
import pandas as pd
pd.core.common.is_list_like = pd.api.types.is_list_like
import pandas_datareader as web
from pandas_datareader import data as pdr
from datetime import datetime
import yfinance as yf

In [ ]:
import warnings
from pandas.errors import Pandas4Warning
warnings.filterwarnings("ignore", category=Pandas4Warning, message=".*inplace.*")

ticker = "AAPL"

# download apple data for a specific daterange
df_share = yf.download("AAPL", start="2000-01-01", end="2024-01-01")
print(df_share.head(3))
print(df_share.tail(3))

In [ ]:
df_share.info()

In [ ]:
df_share["Close"].plot(title=ticker)
plt.show()

## Economics Data from FRED

In [ ]:
from datetime import date
series_code = 'DGS10' # 10-year Treasury Rate
data_source = 'fred' # FED Economic Data Service
start = date(1962, 1, 1)
myData = pdr.DataReader(series_code, data_source, start)

In [ ]:
myData.head()

In [ ]:
myData = myData / 100.
series_name = '10-year Treasury'
myData = myData.rename(columns={series_code: series_name})
myData.plot(title=series_name)
plt.show() 

## Analysing real data

In [ ]:
# ticker = "^GSPC" ## S&P500 
# GSPC : Global Standard & Poor’s Composite
spx = yf.download("^GSPC", start="2020-01-01", end="2024-01-01")
spx.head()

In [ ]:
spx.head()

In [ ]:
spx['Close'].plot(figsize=(8, 5), grid=True);

### A quick note on computation time -- computing daily returns

In [ ]:
len(spx)

In [ ]:
spx.head()

In [ ]:
%%time
spx['ReturnsLoop'] = 0.
for i in range(1, len(spx)):
    spx['ReturnsLoop'][i] = np.log(spx['Close'][i] / spx['Close'][i - 1])

In [ ]:
spx[['Close', 'ReturnsLoop']].head()

In [ ]:
%time spx['Returns'] = np.log(spx['Close'] / spx['Close'].shift(1))
spx[['Close', 'Returns']].head()

In [ ]:
del spx['ReturnsLoop']

In [ ]:
spx = spx.dropna()
spx.head()

In [ ]:
val = df.iloc[0,0]
val

In [ ]:
df[df['SPX'] == val]['DAX'] = 0.
df.head()

In [ ]:
df.loc[df['SPX'] == val, 'DAX'] = 0.
df.head()

**WHAT IS THE PROBLEM?**

### Data analysis

In [ ]:
spx[['Close', 'Returns']].plot(subplots=True, style='b',figsize=(8, 6));

**Remarks:**
- Volatility clustering
- Leverage effect

### Moving averages

In [ ]:
n1, n2 = 24, 252
MA1, MA2 = 'MA' + str(n1) + 'd', 'MA' + str(n2) + 'd'
spx[MA1] = spx['Close'].rolling(window=n1, center = False).mean()
spx[MA2] = spx['Close'].rolling(window=n2, center = False).mean()
spx.tail()

In [ ]:
spx[['Close', MA1, MA2]].plot(figsize=(8, 5), grid=True);

In [ ]:
spx['MovingReturnsVol'] = spx['Returns'].rolling(window=252).std() * np.sqrt(252.)

In [ ]:
spx[['Close', 'Returns', 'MovingReturnsVol']].plot(subplots=True, style='b',figsize=(8, 7), grid=True);

### *** Now do Exercise 3 ***

# The `groupby` command

In [ ]:
import pandas as pd

file_name = "https://github.com/tdhadfield/MATH_70112/raw/main/Class.xlsx"
df = pd.read_excel(file_name)
df.head()

### Note: this is completely random data!

In [ ]:
df.info()

In [ ]:
df["Grade"].mean()

In [ ]:
print(df["Grade"].sum())
print(df["Grade"].sum() / len(df.index))

In [ ]:
df.groupby(['Nationality']).agg({'Grade': 'mean'})

In [ ]:
df.groupby(['Nationality']).agg({'Grade': 'sum'})

In [ ]:
df.groupby(['Nationality', 'Gender']).agg({'Grade': 'mean'})#.sum()

**WHAT IS HAPPENING????**

### *** Now do Exercise 4: groupby on the class grades ***

### *** Now do Exercise 5: `groupby` and reshaping: the chopstick experiment ***

### *** Now do Exercise 6: Merging and concatenating: Amazon options ***

### *** Now do Exercise 7: Data cleaning and filling gaps ***

# Common issues

In [ ]:
xx = [2, 4, 6, '8']
df = pd.DataFrame(xx)
df.columns = ["X"]
df.head()

In [ ]:
df["Mult"] = 2.*df["X"]

In [ ]:
df.info()

In [ ]:
df['X'] = pd.to_numeric(df['X'])
df["Mult"] = 2.*df["X"]

In [ ]:
df.info()

In [ ]:
df.head()

### Working with copies

In [ ]:
N = 5
xx = np.random.standard_normal((N, 5))
df = pd.DataFrame(xx)
df.head()

In [ ]:
df2 = df ### same as df.copy(deep=False)
df3 = df.copy(deep=True)

In [ ]:
i,j = 0,1
df.iat[i,j] = 2.
print(df.iat[i,j], df2.iat[i,j], df3.iat[i,j])

In [ ]:
i,j = 0,1
df3.iat[i,j] = 5.
print(df.iat[i,j], df2.iat[i,j], df3.iat[i,j])

### Filling the void

In [ ]:
file_name = "https://github.com/tdhadfield/MATH_70112/raw/main/EURGBP-2020-01.csv"

df = pd.read_csv(file_name) 
df.head()
df.columns=["Pair", "DateTime", "Bid", "Ask"] 
print(df.head(3))
df = df.drop(columns=['Pair'])
print(df.head(3))
print(df.tail(3))

In [ ]:
df.plot(title="eurgbp")

In [ ]:
import random

nbMissingValues = len(df)//3
N = len(df)
listMissingValues = random.choices(range(N), k=nbMissingValues)
for l in listMissingValues:
    df.iat[l, 1] = np.log(-1.)

In [ ]:
df.head(10)

In [ ]:
df["Bid"].plot()

How do we fill the gaps? 
- `dropna`?
- interpolation? But which one?

In [ ]:
df["Bid"].interpolate(method='linear', axis=0).plot();

### *** Now do Exercise 7: Data cleaning and filling gaps ***

# Extra: Combining DataFrames and creating signals

In [ ]:
import yfinance as yf

aapl = yf.download("AAPL", start="1980-01-01", end="2024-01-31")
coke = yf.download("COKE", start="1980-01-01", end="2024-01-31")

In [ ]:
print(aapl.columns)
aapl.tail(3)

In [ ]:
coke.tail(3)

In [ ]:
df = pd.merge(coke, aapl, left_index=True, right_index=True)
df.head()

In [ ]:
df = pd.merge(coke["Close"], aapl["Close"], left_index=True, right_index=True)
df.columns = ["coke", "aapl"]
df.head()

In [ ]:
df.plot();

#### Normalised versions...

In [ ]:
coke = coke["Close"] 
aapl = aapl["Close"] 

In [ ]:
coke0 = coke.iloc[0]
aapl0 = aapl.iloc[0]

coke_norm = coke / coke0
aapl_norm = aapl / aapl0
df_norm = pd.merge(coke_norm, aapl_norm, left_index=True, right_index=True)
df_norm.columns = ["coke norm", "aapl norm"]
df_norm.plot();

In [ ]:
coke = yf.download("COKE", start="1980-01-01", end="2024-01-31")

In [ ]:
coke["Returns"] = np.log(coke['Close'] / coke['Close'].shift(1))
coke['MovingVol'] = coke['Returns'].rolling(window=252).std() * np.sqrt(252.)
coke = coke[["Close", "Returns", "MovingVol"]]
coke = coke.dropna()
coke["MovingVol"].plot(title="Moving vol");

In [ ]:
print(coke["MovingVol"].max())

In [ ]:
maxVolAllowed = 0.4
coke.loc[coke['MovingVol'] > maxVolAllowed, 'Signal'] = 0.6
coke[["Signal", "MovingVol"]].plot().legend(loc=3);

### *** Now do Exercise 8 (challenge): Combining series and a volatility signal ***

# WARNINGS: Data inconsistencies

- Missing data
- NaN, -1, ...
- Date format
- Blank spaces
- Formats: Int, float, ...